- groupby() use karke data ka summary nikalna

In [2]:
import pandas as pd

df = pd.read_csv(
    '../data/processed/step4_datatypes_fixed.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate']
)
df['LineTotal'] = df['Quantity'] * df['Price']
print(df.shape)

(779495, 14)


Note: groupby() SQL ke GROUP BY jaisa hi hai (jo tum Month 1 mein kar chuke ho) — data ko categories mein baant kar har category ka summary nikalta hai. Basic pattern: split → apply → combine.

1. Basic groupby — single column, single aggregation:

In [3]:
country_revenue = df.groupby('Country')['LineTotal'].sum()
print(country_revenue.sort_values(ascending=False).head(10))

Country
United Kingdom    1.438923e+07
EIRE              6.165705e+05
Netherlands       5.540381e+05
Germany           4.250197e+05
France            3.487690e+05
Australia         1.692835e+05
Spain             1.083325e+05
Switzerland       1.000619e+05
Sweden            9.151582e+04
Denmark           6.858069e+04
Name: LineTotal, dtype: float64


2. Multiple aggregations ek saath — .agg():

In [4]:
country_summary = df.groupby('Country').agg(
    total_revenue=('LineTotal', 'sum'),
    total_orders=('Invoice', 'nunique'),
    total_customers=('Customer ID', 'nunique'),
    avg_order_value=('LineTotal', 'mean')
)
print(country_summary.sort_values('total_revenue', ascending=False).head(10))

                total_revenue  total_orders  total_customers  avg_order_value
Country                                                                      
United Kingdom   1.438923e+07         33546             5353        20.543313
EIRE             6.165705e+05           567                5        39.607538
Netherlands      5.540381e+05           229               22       108.848348
Germany          4.250197e+05           789              107        25.852780
France           3.487690e+05           614               95        25.811794
Australia        1.692835e+05            95               15        94.466217
Spain            1.083325e+05           154               41        29.574799
Switzerland      1.000619e+05            90               22        33.287405
Sweden           9.151582e+04           104               19        69.488094
Denmark          6.858069e+04            43               12        88.149987


Note: .agg() mein tuple (column, function) deke naye named columns bana sakte ho — yeh SQL ke SUM(x) as total_revenue jaisa hi hai.

3. Multiple columns par groupby (composite grouping):

In [5]:
country_month_revenue = df.groupby(['Country', df['InvoiceDate'].dt.to_period('M')])['LineTotal'].sum()
print(country_month_revenue.head(15))

Country    InvoiceDate
Australia  2009-12          271.10
           2010-02         1029.66
           2010-03          429.39
           2010-04          630.95
           2010-05         2371.15
           2010-06         3214.78
           2010-07          686.12
           2010-08          176.00
           2010-09          785.83
           2010-10         2989.15
           2010-11        18245.52
           2010-12          965.35
           2011-01         9017.71
           2011-02        14695.42
           2011-03        17223.99
Name: LineTotal, dtype: float64


4. groupby() ke result par further operations — sirf top N dikhana:

In [6]:
top_customers = df.groupby('Customer ID')['LineTotal'].sum().sort_values(ascending=False).head(10)
print(top_customers)

Customer ID
18102    580987.04
14646    528602.52
14156    313437.62
14911    291420.81
17450    244784.25
13694    195640.69
17511    172132.87
16446    168472.50
16684    147142.77
12415    144458.37
Name: LineTotal, dtype: float64


5. Multiple functions ek column par ek saath (.agg() with list):

In [7]:
customer_stats = df.groupby('Customer ID')['LineTotal'].agg(['sum', 'mean', 'count', 'min', 'max'])
print(customer_stats.head(10))

                  sum         mean  count   min       max
Customer ID                                              
12346        77556.46  2281.072353     34  1.00  77183.60
12347         4921.53    22.169054    222  5.04    249.60
12348         2019.40    39.596078     51  1.00    240.00
12349         4428.69    25.306800    175  6.64    300.00
12350          334.40    19.670588     17  8.50     40.00
12351          300.93    14.330000     21  9.90     23.40
12352         2849.84    27.668350    103  9.90    376.50
12353          406.76    16.948333     24  5.04     39.80
12354         1079.40    18.610345     58  8.50     54.08
12355          947.61    27.074571     35  5.04    120.00


6. groupby() + filtering (jaise SQL ka HAVING):

In [8]:
customer_totals = df.groupby('Customer ID')['LineTotal'].sum()
high_value_customers = customer_totals[customer_totals > 5000]
print(high_value_customers.shape)
print(high_value_customers.sort_values(ascending=False).head(10))

(645,)
Customer ID
18102    580987.04
14646    528602.52
14156    313437.62
14911    291420.81
17450    244784.25
13694    195640.69
17511    172132.87
16446    168472.50
16684    147142.77
12415    144458.37
Name: LineTotal, dtype: float64


7. groupby() + transform() — group ki value ko wapas original DataFrame mein daalna (Window Function jaisa concept, Day 11 yaad hai):

In [9]:
df['customer_total_spend'] = df.groupby('Customer ID')['LineTotal'].transform('sum')
print(df[['Customer ID', 'LineTotal', 'customer_total_spend']].head(10))

   Customer ID  LineTotal  customer_total_spend
0        13085       83.4               2433.28
1        13085       81.0               2433.28
2        13085       81.0               2433.28
3        13085      100.8               2433.28
4        13085       30.0               2433.28
5        13085       39.6               2433.28
6        13085       30.0               2433.28
7        13085       59.5               2433.28
8        13085       30.6               2433.28
9        13085       45.0               2433.28


Note: transform() result ko original row count ke saath return karta hai (SQL Window Function jaisa), jabki normal groupby().sum() rows ko collapse kar deta hai (SQL GROUP BY jaisa).

8. groupby() object ko iterate karna (advanced, kam use hota hai lekin samajhna zaroori):

In [10]:
for country, group in df.groupby('Country'):
    if country in ['Germany', 'France']:
        print(f"{country}: {group.shape[0]} rows, Total revenue: {group['LineTotal'].sum():.2f}")

France: 13512 rows, Total revenue: 348768.96
Germany: 16440 rows, Total revenue: 425019.71


Practice questions:

1. groupby('DayOfWeek') use karke pata karo kaunse din sabse zyada revenue hota hai (pehle DayOfWeek column banao agar nahi hai).

In [13]:
# DayOfWeek column add karo (jaise 'Monday', 'Tuesday', etc.)
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

# Day ke hisaab se total revenue nikal kar descending order mein sort karo
dow_revenue = df.groupby('DayOfWeek')['LineTotal'].sum().sort_values(ascending=False)

print("--- Revenue by Day of the Week ---")
print(dow_revenue)

--- Revenue by Day of the Week ---
DayOfWeek
Thursday     3745783.432
Tuesday      3322830.142
Wednesday    3021043.853
Monday       2778201.566
Friday       2728473.173
Sunday       1768669.052
Saturday        9803.050
Name: LineTotal, dtype: float64


- Note: Agar aapko days ko chronological order (Monday to Sunday) mein dekhna ho, toh aap pd.Categorical ya custom sorting ka use kar sakte ho, lekin upar wala code direct yeh bata dega ki kis din sabse zyada revenue hua hai.

2. .agg() se ek summary banao jo har Customer ID ka Recency (max InvoiceDate se distance — hint: pehle max date nikalo), Frequency (unique invoices), aur Monetary (total LineTotal) ek saath dikhaye — yeh Bonus Day 35B ka hi practice hai, dobara khud se try karo.

In [14]:
# Dataset ki sabse aakhri date nikalo
max_date = df['InvoiceDate'].max()

# .agg() ka use karke RFM summary banao
rfm_summary = df.groupby('Customer ID').agg(
    Recency=('InvoiceDate', lambda x: (max_date - x.max()).days),
    Frequency=('Invoice', 'nunique'),
    Monetary=('LineTotal', 'sum')
)

print("\n--- Customer RFM Summary ---")
print(rfm_summary.head(10))


--- Customer RFM Summary ---
             Recency  Frequency  Monetary
Customer ID                              
12346            325         12  77556.46
12347              1          8   4921.53
12348             74          5   2019.40
12349             18          4   4428.69
12350            309          1    334.40
12351            374          1    300.93
12352             35         10   2849.84
12353            203          2    406.76
12354            231          1   1079.40
12355            213          2    947.61
